In [ ]:
from transformers import AutoTokenizer, AutoConfig

TEACHER = "Qwen/Qwen3.8-27B"
STUDENT = "Qwen/Qwen3.5-0.8B"

teacher_tokenizer = AutoTokenizer.from_pretrained(TEACHER)
student_tokenizer = AutoTokenizer.from_pretrained(STUDENT)

In [3]:
print(len(teacher_tokenizer), len(student_tokenizer), len(teacher_tokenizer) == len(student_tokenizer))

248077 248077 True


In [4]:
teacher_vocab =  teacher_tokenizer.get_vocab()
student_vocab =  student_tokenizer.get_vocab()

print(type(teacher_vocab))
print(teacher_vocab == student_vocab) # each key, value pair match

<class 'dict'>
True


In [5]:
samples = [
    "The capital of France is Paris.",
    "def f(x):\n    return x ** 2\n",   
    "Solve for x: 3x + 7 = 22", 
    "1234567890",
]

for sample in samples:
    a = teacher_tokenizer.encode(sample)
    b = student_tokenizer.encode(sample)
    print(a)
    print(b)
    print(f"match = {a == b}")


[760, 6511, 314, 9338, 369, 11751, 13]
[760, 6511, 314, 9338, 369, 11751, 13]
match = True
[727, 281, 2007, 1590, 198, 262, 460, 830, 2972, 220, 17, 198]
[727, 281, 2007, 1590, 198, 262, 460, 830, 2972, 220, 17, 198]
match = True
[50, 3815, 364, 830, 25, 220, 18, 87, 478, 220, 22, 283, 220, 17, 17]
[50, 3815, 364, 830, 25, 220, 18, 87, 478, 220, 22, 283, 220, 17, 17]
match = True
[16, 17, 18, 19, 20, 21, 22, 23, 24, 15]
[16, 17, 18, 19, 20, 21, 22, 23, 24, 15]
match = True


In [9]:
print(teacher_tokenizer.chat_template)

{%- set image_count = namespace(value=0) %}
{%- set video_count = namespace(value=0) %}
{%- macro render_content(content, do_vision_count, is_system_content=false) %}
    {%- if content is string %}
        {{- content }}
    {%- elif content is iterable and content is not mapping %}
        {%- for item in content %}
            {%- if 'image' in item or 'image_url' in item or item.type == 'image' %}
                {%- if is_system_content %}
                    {{- raise_exception('System message cannot contain images.') }}
                {%- endif %}
                {%- if do_vision_count %}
                    {%- set image_count.value = image_count.value + 1 %}
                {%- endif %}
                {%- if add_vision_id %}
                    {{- 'Picture ' ~ image_count.value ~ ': ' }}
                {%- endif %}
                {{- '<|vision_start|><|image_pad|><|vision_end|>' }}
            {%- elif 'video' in item or item.type == 'video' %}
                {%- if is_s

In [11]:
print(student_tokenizer.chat_template)

{%- set image_count = namespace(value=0) %}
{%- set video_count = namespace(value=0) %}
{%- macro render_content(content, do_vision_count, is_system_content=false) %}
    {%- if content is string %}
        {{- content }}
    {%- elif content is iterable and content is not mapping %}
        {%- for item in content %}
            {%- if 'image' in item or 'image_url' in item or item.type == 'image' %}
                {%- if is_system_content %}
                    {{- raise_exception('System message cannot contain images.') }}
                {%- endif %}
                {%- if do_vision_count %}
                    {%- set image_count.value = image_count.value + 1 %}
                {%- endif %}
                {%- if add_vision_id %}
                    {{- 'Picture ' ~ image_count.value ~ ': ' }}
                {%- endif %}
                {{- '<|vision_start|><|image_pad|><|vision_end|>' }}
            {%- elif 'video' in item or item.type == 'video' %}
                {%- if is_s

In [ ]:
print(teacher_tokenizer.chat_template == student_tokenizer.chat_template) # I will just use the teacher's chat_template

False


Just measuring hte dataset tokens lengths

In [ ]:
import json
from pathlib import Path
from transformers import AutoTokenizer

teacher_tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3.8-27B")

In [22]:
prompts_path = Path("shared_code/prompts_train.jsonl")
rows = [json.loads(line) for line in open(prompts_path, encoding="utf-8")]
print("rows", len(rows))
print(rows[0])

messages = [{"role": "user", "content": rows[0]["prompt"]}]
formatted = teacher_tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    reasoning_effort="low",
)
print(len(formatted))
print(repr(formatted[:400]))
print(repr(formatted[-200:]))


rows 47473
{'prompt': "BMW was established as a business entity following a restructuring of the Rapp Motorenwerke aircraft manufacturing firm in 1917. After the end of World War I in 1918, BMW was forced to cease aircraft-engine production by the terms of the Versailles Armistice Treaty. The company consequently shifted to motorcycle production as the restrictions of the treaty started to be lifted in 1923, followed by automobiles in 1928–29.\nPick up the MINI Hardtop 2 Door, the Hardtop 4 Door, or the MINI Countryman with available ALL4 all-wheel drive for no monthly payments until 2016. Find one near you.\nThe Bose SoundLink Mini II is a small Bluetooth speaker. It isn't small enough that you'll carry it with you every day, and you’re definitely not going to fit it in a pocket. However, it's about as small as a speaker can get without compromising on sound quality.\nThe good news is that Bose has addressed virtually all of the hardware complaints about the first version. First, rath

In [23]:
lengths = []
for row in rows: 
    text = teacher_tokenizer.apply_chat_template(
        [{"role": "user", "content": row["prompt"]}],
        tokenize=False,
        add_generation_prompt=True,
        reasoning_effort="low",
    )
    tokens_len = len(teacher_tokenizer.encode(text))
    domain = row["domain"]
    lengths.append((domain, tokens_len))


In [31]:
import statistics, collections
by_domain = collections.defaultdict(list)
for domain, n in lengths:
    by_domain[domain].append(n)

total = 0
for domain, values in by_domain.items():
    print(domain, "mean", int(statistics.mean(values)), "median", int(statistics.median(values)),
          "min", min(values), "max", max(values), "sum", sum(values))
    total += sum(values)
print("\nTotal Input Tokens", total)

rag mean 723 median 719 min 382 max 1299 sum 5784975
math mean 99 median 96 min 52 max 252 sum 744098
chat mean 138 median 121 min 46 max 832 sum 1109906
qa mean 50 median 50 min 47 max 65 sum 403031
summarization mean 824 median 790 min 239 max 1992 sum 6596965
translation mean 78 median 74 min 53 max 247 sum 628426

Total Input Tokens 15267401


I will be training with input prompts with the total of 15.2M tokens